Cell 1 — Parameters

In [1]:
# Fabric notebook parameters
config_table_name = "pipeline.ingest_config"
mapping_base_path = "/mappings/bronze_silver/"
run_env = "prod"
force_full_reload = False

Cell 2 — Imports and Setup

In [2]:
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, DecimalType
from pyspark.sql.utils import AnalysisException
from typing import List, Dict, Optional
from datetime import datetime
import json

try:
    spark
except NameError:
    spark = SparkSession.builder.appName("nb_ingest_bronze_silver").getOrCreate()

INVALID_TABLE = "log.invalid_record"
PIPELINE_NAME = "bronze_to_silver"
run_timestamp = datetime.utcnow()

KeyboardInterrupt: 

Cell 3 — Step 1: Read Config Table

In [3]:
def read_config_table(config_table_name: str):
    try:
        df = spark.table(config_table_name)
    except AnalysisException as e:
        raise ValueError(f"Cannot read config table '{config_table_name}': {e}")

    rows = df.filter(F.col("is_active") == True).collect()
    if not rows:
        raise ValueError(f"Config table '{config_table_name}' returned 0 active rows.")

    print(f"[STEP 1] Found {len(rows)} active pipeline row(s).")
    return rows

Cell 4 — Step 2: Load JSON Mapping File

In [4]:
def validate_mapping_object(mapping: dict) -> dict:
    required_keys = {"source_table", "target_table", "columns"}
    missing = required_keys - set(mapping.keys())
    if missing:
        raise ValueError(f"Mapping JSON missing required keys: {sorted(missing)}")

    if not isinstance(mapping["columns"], list) or not mapping["columns"]:
        raise ValueError("Mapping JSON 'columns' must be a non-empty list")

    for i, c in enumerate(mapping["columns"]):
        if "target" not in c:
            raise ValueError(f"Mapping JSON column #{i} missing 'target'")
        if "expression" not in c:
            raise ValueError(f"Mapping JSON column #{i} missing 'expression'")

    return mapping

def load_json_mapping_file(mapping_json_path: str) -> dict:
    try:
        raw_text = spark.read.text(mapping_json_path).collect()
        if not raw_text:
            raise ValueError(f"Mapping file is empty: {mapping_json_path}")
        content = "\n".join([r["value"] for r in raw_text])
        mapping = json.loads(content)
    except AnalysisException as e:
        raise ValueError(f"Mapping file not found or unreadable: {mapping_json_path} | {e}")
    except json.JSONDecodeError as e:
        raise ValueError(f"Malformed JSON in mapping file: {mapping_json_path} | {e}")

    return validate_mapping_object(mapping)

Cell 5 — Step 3: Read Bronze Source Table

In [5]:
def read_bronze_table(source_table: str) -> tuple[DataFrame, int]:
    try:
        df_bronze = spark.table(source_table)
    except AnalysisException as e:
        raise ValueError(f"Bronze table not found: {source_table} | {e}")

    rows_read = df_bronze.count()
    print(f"[STEP 3] Read {rows_read:,} row(s) from {source_table}")
    return df_bronze, rows_read

Cell 6 — Step 4: Apply Column Transformations

In [6]:
def apply_column_transformations(df_bronze: DataFrame, mapping: dict, null_defaults: Optional[dict] = None) -> DataFrame:
    null_defaults = null_defaults or {"string": "", "int": 0, "double": 0.0, "decimal": 0.0}
    bronze_cols = set(df_bronze.columns)
    select_exprs = []
    mapped_cols = []
    defaulted_cols = []

    for item in mapping["columns"]:
        target = item["target"]
        expr = item["expression"]

        if expr is None:
            select_exprs.append(F.lit(None).alias(target))
            defaulted_cols.append(target)
            continue

        if expr in bronze_cols:
            select_exprs.append(F.col(expr).alias(target))
            mapped_cols.append(f"{expr}->{target}")
        else:
            try:
                select_exprs.append(F.expr(expr).alias(target))
                mapped_cols.append(f"expr:{expr}->{target}")
            except Exception as e:
                raise ValueError(
                    f"Cannot build expression for target '{target}' using '{expr}' "
                    f"from source '{mapping['source_table']}': {e}"
                )

    print(f"[STEP 4] Mapped columns: {mapped_cols}")
    if defaulted_cols:
        print(f"[STEP 4] Default/null columns: {defaulted_cols}")

    return df_bronze.select(*select_exprs)

Cell 7 — Step 5: Data Quality Validation Engine

In [7]:
def add_failure_reason(df: DataFrame) -> DataFrame:
    return df.withColumn("__dq_failure_reason", F.lit(None).cast(StringType()))

def append_reason(df: DataFrame, fail_cond, reason: str) -> DataFrame:
    return df.withColumn(
        "__dq_failure_reason",
        F.when(
            fail_cond,
            F.when(F.col("__dq_failure_reason").isNull(), F.lit(reason))
             .otherwise(F.concat(F.col("__dq_failure_reason"), F.lit("|"), F.lit(reason)))
        ).otherwise(F.col("__dq_failure_reason"))
    )

def check_not_null(column: str):
    return F.col(column).isNull()

def check_in_set(column: str, allowed_values: list):
    return ~F.col(column).isin(allowed_values)

def check_positive(column: str):
    return F.col(column).isNull() | (F.col(column) <= 0)

def check_non_negative(column: str):
    return F.col(column) < 0

def check_date_valid(column: str):
    return F.col(column).isNotNull() & F.to_date(F.col(column)).isNull()

def check_date_order(start_col: str, end_col: str, inclusive: bool = False):
    if inclusive:
        return F.col(start_col).isNotNull() & F.col(end_col).isNotNull() & (F.col(start_col) > F.col(end_col))
    return F.col(start_col).isNotNull() & F.col(end_col).isNotNull() & (F.col(start_col) >= F.col(end_col))

def check_timestamp_not_future(column: str):
    return F.col(column).isNotNull() & (F.to_date(F.col(column)) > F.current_date())

def check_date_not_future_or_null(column: str):
    return F.col(column).isNull() | (F.to_date(F.col(column)) > F.current_date())

def check_pattern(column: str, pattern: str):
    return F.col(column).isNotNull() & (~F.col(column).rlike(pattern))

def run_dq_validation(input_df: DataFrame, dq_rules: List[Dict], quarantine_table: str = INVALID_TABLE):
    if not dq_rules:
        df_valid = input_df
        df_rejected = input_df.limit(0).withColumn("__dq_failure_reason", F.lit(None).cast(StringType()))
        return df_valid, df_rejected, 0

    work = add_failure_reason(input_df)

    unique_rules = [r for r in dq_rules if r["rule"] == "unique"]
    for rule in unique_rules:
        col = rule["column"]
        dup_keys = (
            work.groupBy(col)
            .count()
            .filter((F.col(col).isNotNull()) & (F.col("count") > 1))
            .select(col)
        )
        dup_vals = [r[col] for r in dup_keys.collect()]
        if dup_vals:
            work = append_reason(work, F.col(col).isin(dup_vals), f"unique:{col}")

    for rule in dq_rules:
        rule_type = rule["rule"]
        col = rule.get("column")
        reason = f"{rule_type}:{col}" if col else rule_type

        if rule_type == "not_null":
            work = append_reason(work, check_not_null(col), reason)
        elif rule_type == "in_set":
            work = append_reason(work, check_in_set(col, rule["values"]), reason)
        elif rule_type == "positive":
            work = append_reason(work, check_positive(col), reason)
        elif rule_type == "non_negative":
            work = append_reason(work, check_non_negative(col), reason)
        elif rule_type == "date_valid":
            work = append_reason(work, check_date_valid(col), reason)
        elif rule_type == "date_order":
            work = append_reason(work, check_date_order(rule["start_col"], rule["end_col"], rule.get("inclusive", False)), reason)
        elif rule_type == "timestamp_not_future":
            work = append_reason(work, check_timestamp_not_future(col), reason)
        elif rule_type == "date_not_future_or_null":
            work = append_reason(work, check_date_not_future_or_null(col), reason)
        elif rule_type == "pattern":
            work = append_reason(work, check_pattern(col, rule["pattern"]), reason)

    df_valid = work.filter(F.col("__dq_failure_reason").isNull()).drop("__dq_failure_reason")
    df_rejected = work.filter(F.col("__dq_failure_reason").isNotNull()).withColumn("__quarantined_at", F.current_timestamp())
    dq_failure_count = df_rejected.count()

    if dq_failure_count > 0:
        try:
            df_rejected.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(quarantine_table)
            print(f"[STEP 5] Wrote {dq_failure_count:,} rejected row(s) to {quarantine_table}")
        except Exception as e:
            print(f"[STEP 5] Quarantine write failed: {e}")

    return df_valid, df_rejected, dq_failure_count

NameError: name 'INVALID_TABLE' is not defined

Cell 8 — Step 5: DQ Rules for bronze.agent

In [ ]:
agent_dq_rules = [
    {"column": "agent_id", "rule": "not_null"},
    {"column": "agent_name", "rule": "not_null"},
]

Cell 9 — Step 7: Audit Log Wrapper

In [ ]:
def write_audit_log(
    audit_log_fn,
    pipeline_name: str,
    source_table: str,
    target_table: str,
    run_timestamp,
    rows_read: int,
    rows_valid: int,
    rows_rejected: int,
    rows_merged: int,
    status: str,
    error_message: Optional[str] = None,
):
    try:
        audit_log_fn(
            pipeline_name=pipeline_name,
            source_table=source_table,
            target_table=target_table,
            run_timestamp=run_timestamp,
            rows_read=rows_read,
            rows_valid=rows_valid,
            rows_rejected=rows_rejected,
            rows_merged=rows_merged,
            status=status,
            error_message=error_message,
        )
    except Exception as e:
        print(f"[AUDIT] Failed to write audit log: {e}")

In [ ]:

def audit_log(
    pipeline_name: str,
    source_table: str,
    target_table: str,
    run_timestamp,
    rows_read: int,
    rows_valid: int,
    rows_rejected: int,
    rows_merged: int,
    status: str,
    error_message: Optional[str] = None,
):
    print(
        f"[AUDIT] pipeline={pipeline_name} source={source_table} target={target_table} "
        f"read={rows_read} valid={rows_valid} rejected={rows_rejected} merged={rows_merged} "
        f"status={status} error={error_message}"
    )


Cell 10 — Step 6: Dynamic MERGE INTO Silver

In [ ]:
def merge_into_silver(df_valid: DataFrame, target_table: str, merge_keys: List[str], force_full_reload: bool = False) -> int:
    if not merge_keys:
        raise ValueError("merge_keys must not be empty")

    rows_merged = df_valid.count()

    if force_full_reload:
        df_valid.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(target_table)
        print(f"[STEP 6] Full reload completed for {target_table}")
        return rows_merged

    source_alias = "s"
    target_alias = "t"
    join_cond = " AND ".join([f"{target_alias}.{k} = {source_alias}.{k}" for k in merge_keys])
    update_set = ", ".join([f"{target_alias}.{c} = {source_alias}.{c}" for c in df_valid.columns])
    insert_cols = ", ".join(df_valid.columns)
    insert_vals = ", ".join([f"{source_alias}.{c}" for c in df_valid.columns])

    df_valid.createOrReplaceTempView("stg_merge_source")

    merge_sql = f"""
    MERGE INTO {target_table} AS {target_alias}
    USING stg_merge_source AS {source_alias}
    ON {join_cond}
    WHEN MATCHED THEN UPDATE SET {update_set}
    WHEN NOT MATCHED THEN INSERT ({insert_cols})
    VALUES ({insert_vals})
    """

    spark.sql(merge_sql)
    print(f"[STEP 6] MERGE completed for {target_table}")
    return rows_merged

Cell 11 — Main Orchestrator for Each Config Row

In [ ]:
def process_one_pipeline_row(config_row, null_defaults=None, audit_log_fn=None):
    rows_read = 0
    rows_valid = 0
    rows_rejected = 0
    rows_merged = 0
    status = "FAILED"
    error_message = None

    try:
        mapping = load_json_mapping_file(config_row["mapping_json_path"])
        source_table = mapping["source_table"]
        target_table = mapping["target_table"]

        df_bronze, rows_read = read_bronze_table(source_table)
        df_mapped = apply_column_transformations(df_bronze, mapping, null_defaults=null_defaults)

        if target_table == "silver.agent":
            dq_rules = agent_dq_rules
        else:
            dq_rules = []

        df_valid, df_rejected, rows_rejected = run_dq_validation(
            df_mapped,
            dq_rules,
            quarantine_table=f"silver_quarantine.{target_table.split('.')[-1]}"
        )
        rows_valid = df_valid.count()

        if rows_valid == 0:
            status = "WARNING"
            print(f"[MAIN] All rows rejected for {target_table}; skipping MERGE.")
        else:
            rows_merged = merge_into_silver(
                df_valid=df_valid,
                target_table=target_table,
                merge_keys=config_row["merge_keys"],
                force_full_reload=force_full_reload
            )
            status = "SUCCESS" if rows_rejected == 0 else "WARNING"

    except Exception as e:
        error_message = str(e)
        status = "FAILED"
        print(f"[MAIN] Failed for source={config_row['source_table']} target={config_row['target_table']}: {error_message}")

    finally:
        if audit_log_fn is not None:
            write_audit_log(
                audit_log_fn=audit_log_fn,
                pipeline_name=PIPELINE_NAME,
                source_table=config_row["source_table"],
                target_table=config_row["target_table"],
                run_timestamp=run_timestamp,
                rows_read=rows_read,
                rows_valid=rows_valid,
                rows_rejected=rows_rejected,
                rows_merged=rows_merged,
                status=status,
                error_message=error_message,
            )

    return {
        "source_table": config_row["source_table"],
        "target_table": config_row["target_table"],
        "rows_read": rows_read,
        "rows_valid": rows_valid,
        "rows_rejected": rows_rejected,
        "rows_merged": rows_merged,
        "status": status,
        "error_message": error_message,
    }

Cell 12 — Step 1 to 7 Runner

In [ ]:
config_rows = read_config_table(config_table_name)

results = []
for row in config_rows:
    config_row = row.asDict(recursive=True)
    result = process_one_pipeline_row(
        config_row=config_row,
        null_defaults={"string": "", "int": 0, "double": 0.0, "decimal": 0.0},
        audit_log_fn=audit_log
    )
    results.append(result)

summary_df = spark.createDataFrame(results)
display(summary_df)